In [ ]:
#!uv venv /tmp/venv/vllm-ocr-env --allow-existing
#!source /tmp/venv/vllm-ocr-env/bin/activate

!uv pip install -U vllm pyngrok pymupdf kaggle \
    --system --break-system-packages \
    --pre --extra-index-url https://wheels.vllm.ai/nightly \
    --extra-index-url https://download.pytorch.org/whl/cu130 \
    --index-strategy unsafe-best-match

In [ ]:
try:
    import torch
    if torch.cuda.is_available():
        _ = torch.zeros((1,), device="cuda")
        print(f"[Main] GPU activated success: {torch.cuda.get_device_name(0)}")
    else:
        print("[Main] No usable GPU, check kernel-metadata.json")
except Exception as e:
    print(f"GPU exception: {e}")

In [ ]:
!mkdir -p /kaggle/working/logs
import subprocess
import time
import os
import sys

# 启动 vllm 服务进程
model_path = "zai-org/GLM-OCR"
nvidia_lib_path = "/usr/local/nvidia/lib64"
env = os.environ.copy()
if "LIBRARY_PATH" in env:
    env["LIBRARY_PATH"] = f"{env['LIBRARY_PATH']}:{nvidia_lib_path}"
else:
    env["LIBRARY_PATH"] = nvidia_lib_path

uv_python = "python"
env["PYTHONUNBUFFERED"] = "1"
env["CUDA_VISIBLE_DEVICES"] = "0"
env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
vllm_log_file_0 = open("/kaggle/working/vllm_0.log", "w", encoding="utf-8")
ocr_process_0 = subprocess.Popen([
    uv_python, "-m", "vllm.entrypoints.openai.api_server",
    "--model", model_path,
    "--allowed-local-media-path", "/",
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", "0.85",
    "--served-model-name", "GLM-OCR",
    "--max-model-len", "8192",
    "--port", "8000"
], env=env,
   stdout=vllm_log_file_0,
   stderr=subprocess.STDOUT)

print("ocr 0 server is starting.")

In [ ]:
import subprocess
import time
import os
import sys

# 启动 vllm 服务进程
model_path = "zai-org/GLM-OCR"
nvidia_lib_path = "/usr/local/nvidia/lib64"
env = os.environ.copy()
if "LIBRARY_PATH" in env:
    env["LIBRARY_PATH"] = f"{env['LIBRARY_PATH']}:{nvidia_lib_path}"
else:
    env["LIBRARY_PATH"] = nvidia_lib_path

print(env)
uv_python = "python"
env["PYTHONUNBUFFERED"] = "1"
env["CUDA_VISIBLE_DEVICES"] = "1"
env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
vllm_log_file_1 = open("/kaggle/working/vllm_1.log", "w", encoding="utf-8")
ocr_process_1 = subprocess.Popen([
    uv_python, "-m", "vllm.entrypoints.openai.api_server",
    "--model", model_path,
    "--allowed-local-media-path", "/",
    "--tensor-parallel-size", "1",
    "--gpu-memory-utilization", "0.85",
    "--served-model-name", "GLM-OCR",
    "--max-model-len", "8192",
    "--port", "8001"
], env=env,
   stdout=vllm_log_file_1,
   stderr=subprocess.STDOUT)

print("ocr 1 server is starting...")

In [ ]:
!rm -rf /tmp/chronicles-extractor
!git clone https://github.com/sunchaobo/chronicles-extractor.git \
    /tmp/chronicles-extractor

!rm -rf /tmp/pages
!python /tmp/chronicles-extractor/src/split_pdf.py \
    --pdf /kaggle/input/datasets/chaobosun/pages-to-ocr/original.pdf \
    --pages-dir /tmp/pages

In [ ]:
import os
import sys
import time
import requests


def is_started(port):
    url = f"http://localhost:{port}/health"
    try:
        res = requests.get(url)
        if res.status_code == 200:
            print(f"vLLM at ${port} server Succeeded.")
            return True
    except:
        return False
    
    return False


print(
    "Waiting for vLLM services to start... (checking https.)",
    flush=True,
)

start_time = time.time()
TIMEOUT = 300  # 5分钟超时检测
check_count = 0

while True:
    v0_ready = is_started("8000")
    v1_ready = is_started("8001")

    if v0_ready and v1_ready:
        print("\nBoth vLLM instances started successfully!", flush=True)
        break

    elapsed = int(time.time() - start_time)
    check_count += 1

    # 每 15 秒（循环 3 次）打出一条带换行的日志，保证在 push 模式下能被 capture
    if check_count % 3 == 1:
        print(
            f"[Check {elapsed}s] vLLM_0: {'READY' if v0_ready else 'BOOTING'} | vLLM_1: {'READY' if v1_ready else 'BOOTING'}",
            flush=True,
        )
        sys.stdout.flush()
        os.system("tail -n 5 /kaggle/working/vllm_0.log")
        os.system("tail -n 5 /kaggle/working/vllm_1.log")

    if elapsed > TIMEOUT:
        print("\nTimeout waiting for vLLM to start!", flush=True)

    time.sleep(5)

In [ ]:
!python /tmp/chronicles-extractor/src/ocr_glm.py \
    --pages-dir /tmp/pages \
    --results-dir /kaggle/working/results \
    --api-base http://localhost:8001/v1 http://localhost:8000/v1 \
    --workers 4

In [ ]:
!tar -czvf /kaggle/working/results.tar.gz -C /kaggle/working results
!rm -rf /kaggle/working/results